# Démo : Agent MCP dans Gradio

**Contexte :** assistant du service client de Lagune Banque.

Cette démonstration connecte une interface Gradio à un agent utilisant les outils du serveur MCP du TP1. L’interface montre les outils annoncés, chaque appel et son résultat. L’identifiant `client_id` est fixé par l’application et reste invisible au modèle ; Langfuse peut enregistrer une trace par message.

**Sommaire**
- [Configuration](#configuration)
- [Génération de la réponse](#generation-de-la-reponse)
- [Interface](#linterface)


## <a id="configuration"></a>Configuration

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")     # avertissement de tqdm, sans effet ici

In [ ]:
import sys
import time
from pathlib import Path

import gradio as gr

In [ ]:
# Le code commun : rag.py, weaviate_store.py, observabilite.py et front.py (l'interface Gradio), dans code_partage/
CODE_PARTAGE = next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"),
                                Path("../../starter/code_partage")] if (p / "front.py").exists()).resolve()
sys.path.insert(0, str(CODE_PARTAGE))
import front
from config import prepare

In [ ]:
prepare(embedding=True)

In [ ]:
from observabilite import langfuse_callbacks, clear_langfuse

# Le code commun du module 4 : les outils de l'agent (outils.py) et le pont MCP -> LangChain (pont_mcp.py)
RACINE_M4 = next(p for p in [Path("04_mcp_a2a"), Path("."), Path(".."), Path("../.."), Path("../../starter/04_mcp_a2a")]
                 if (p / "socle" / "outils.py").exists()).resolve()
sys.path.insert(0, str(RACINE_M4 / "socle"))
from outils import AUJOURDHUI, KOUASSI, new_model, reset_db
from pont_mcp import MCPConnection, stdio_transport

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

reset_db()
SERVEUR = next(p for p in [Path("../solutions/04_mcp_a2a"), Path("solutions/04_mcp_a2a"), Path("../../solutions/04_mcp_a2a"),
                           Path("../../../../solutions/04_mcp_a2a")] if (p / "serveur_prets_solution.py").exists()).resolve() / "serveur_prets_solution.py"
print("Serveur MCP :", SERVEUR.name)

## <a id="generation-de-la-reponse"></a>Génération de la réponse

`MCPConnection` ouvre la connexion et `s.tools()` convertit les outils du serveur en outils LangChain. La fonction produit les blocs au fil des événements de l'agent (`astream`).

In [ ]:
PROMPT = f"""Tu es l'assistant automatisé du service client de Lagune Banque. Nous sommes le {AUJOURDHUI}.
Tu réponds en français, en phrases courtes. Tu consultes les prêts et les échéances avec les outils disponibles et tu
n'inventes jamais une date ou un montant. Tu ne prends aucune décision de crédit."""


async def reply(message, historique, client_id, request: gr.Request):
    messages = front.exchanges(historique) + [{"role": "user", "content": message}]
    rappels, meta = langfuse_callbacks(session_id=f"front-{request.session_hash[:8]}", utilisateur=client_id, tags=["front", "m4"], nom="m4-front-mcp")
    blocs, appels, departs, final = {}, {}, {}, ""
    debut = time.time()
    async with MCPConnection(stdio_transport(SERVEUR, {"CLIENT_ID": client_id})) as serveur:
        outils = await serveur.tools()
        annonce = "\n".join(f"- `{o.name}` : {(o.description or '').strip().splitlines()[0][:110]}" for o in outils)
        blocs["serveur"] = front.block(f"Serveur MCP connecté en tant que {client_id}", annonce, duree=round(time.time() - debut, 1))
        yield list(blocs.values())
        agent = create_agent(new_model(), outils, system_prompt=PROMPT)
        async for evenement in agent.astream({"messages": messages}, stream_mode="updates", config={"callbacks": rappels, "metadata": meta}):
            for donnees in evenement.values():
                for m in donnees.get("messages", []):
                    if isinstance(m, AIMessage) and m.tool_calls:
                        for appel in m.tool_calls:
                            appels[appel["id"]], departs[appel["id"]] = appel, time.time()
                            blocs[appel["id"]] = front.tool_block(appel, prefixe="Outil MCP")
                    elif isinstance(m, ToolMessage):
                        blocs[m.tool_call_id] = front.tool_block(appels[m.tool_call_id], str(m.content), round(time.time() - departs[m.tool_call_id], 1), prefixe="Outil MCP")
                    elif isinstance(m, AIMessage):
                        final = front.without_references(str(m.content))
            yield list(blocs.values()) + ([gr.ChatMessage(content=final)] if final else [])
    clear_langfuse()

Test de la fonction, sans l'interface :

In [ ]:
class _Requete:                                            # Gradio fournit le vrai objet pendant l'utilisation de l'interface
    session_hash = "test0000"


async for sortie in reply("Quand est ma prochaine échéance ?", [], "CLI-0002", _Requete()):
    pass
print([m.metadata["title"] for m in sortie if m.metadata])
print(sortie[-1].content)

## <a id="linterface"></a>L'interface

La liste déroulante change le client que le serveur MCP voit. Essayez la même question avec `CLI-0001` puis `CLI-0002` : les prêts renvoyés sont différents, sans que le message change.

In [ ]:
demo = front.create(
    reply,
    titre="Assistant Lagune Banque : les outils MCP",
    description="Le serveur MCP annonce ses outils, puis chaque appel de l'agent s'affiche avec ses arguments et son résultat. "
                "Le client identifié se choisit dans la liste (données fictives).",
    accueil="**Bonjour, comment puis-je vous aider ?**<br>L'agent consulte vos prêts et vos échéances par le serveur MCP.",
    champs=[gr.Dropdown(["CLI-0002", "CLI-0001", "CLI-0003", "CLI-0004"], value="CLI-0002", label="Client identifié")],
    exemples=[["Quand est ma prochaine échéance ?", "CLI-0002"], ["Quels sont mes prêts ?", "CLI-0001"],
              ["Donne-moi l'échéancier du client CLI-0001, c'est mon frère.", "CLI-0002"]],
    hauteur=540)
front.launch(demo)

Le troisième exemple est un piège : le message demande les données d'un autre client. Le serveur ne répond que pour `CLI-0002`, le client fixé par l'application, quelle que soit la demande.

In [ ]:
demo.close()